# Building a text classifier with Differential Privacy

In [ ]:
! pip install pandas
! pip install transformers datasets peft
! pip install trl kagglehub  torch
! pip install scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 1.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 85.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 132.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.8/347.8 kB 116.9 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 6.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 45.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB

In this tutorial, we will train a text classifier with Differential Privacy by taking a model pre-trained on public text data and fine-tuning it for a different task.

When training a model with differential privacy, we almost always face a trade-off between model size and accuracy on the task. The exact details depend on the problem, but a rule of thumb is that the fewer parameters the model has, the easier it is to get good performance with DP.

Most state-of-the-art NLP models are quite deep and large (e.g. [BERT-base](https://github.com/google-research/bert) has over 100M parameters), which makes the task of training text models on private datasets rather challenging.

One way of addressing this problem is to divide the training process into two stages. First, we will pre-train the model on a public dataset, exposing the model to generic text data. Assuming that the generic text data is public, we will not be using differential privacy at this step. Then, we freeze most of the layers, leaving only a few upper layers to be trained on the private dataset using DP-SGD. This way we can get the best of both worlds - we have a deep and powerful text understanding model, while only training a small number of parameters with differentially private algorithm.

In this tutorial, we will take the pre-trained [BERT-base](https://github.com/google-research/bert) model and fine-tune it to recognize textual entailment on the [SNLI](https://nlp.stanford.edu/projects/snli/) dataset.

We further demonstrate fine-tuning results with

- Ghost Clipping DP-SGD, a memory-efficient implementation of DP-SGD, which enables the use of large batch sizes.
- LoRA (low-rank adaptation), a method for parameter-efficienct fine-tuning which can be used in conjucture with DP-SGD to further reduce the number of trainable parameters

## Dataset

First, we need to download the dataset (we'll use Stanford NLP mirror)

In [ ]:
STANFORD_SNLI_URL = "https://nlp.stanford.edu/projects/snli/snli_1.0.zip"
DATA_DIR = "data"

In [ ]:
import zipfile
import urllib.request
import os

import warnings
warnings.simplefilter("ignore")

def download_and_extract(dataset_url, data_dir):
    print("Downloading and extracting ...")
    filename = "snli_1.0.zip"
    urllib.request.urlretrieve(dataset_url, filename)
    with zipfile.ZipFile(filename) as zip_ref:
        zip_ref.extractall(data_dir)
    os.remove(filename)
    print("Completed!")

download_and_extract(STANFORD_SNLI_URL, DATA_DIR)

Completed!


The dataset comes in two formats (`tsv` and `json`) and has already been split into train/dev/test. Let’s verify that’s the case.

In [ ]:
snli_folder = os.path.join(DATA_DIR, "snli_1.0")
os.listdir(snli_folder)

['snli_1.0_train.txt',
 'snli_1.0_train.jsonl',
 'snli_1.0_test.txt',
 'snli_1.0_test.jsonl',
 'snli_1.0_dev.txt',
 'snli_1.0_dev.jsonl',
 'README.txt',
 'Icon\r',
 '.DS_Store']

Let's now take a look inside. [SNLI dataset](https://nlp.stanford.edu/projects/snli/) provides ample syntactic metadata, but we'll only use raw input text. Therefore, the only fields we're interested in are **sentence1** (premise), **sentence2** (hypothesis), and **gold_label** (label chosen by the majority of annotators).

The label defines the relation between premise and hypothesis: either *contradiction*, *neutral*, or *entailment*.

In [ ]:
import pandas as pd
train_path =  os.path.join(snli_folder, "snli_1.0_train.txt")
dev_path = os.path.join(snli_folder, "snli_1.0_dev.txt")

df_train = pd.read_csv(train_path, sep='\t')
df_test = pd.read_csv(dev_path, sep='\t')

df_train[['sentence1', 'sentence2', 'gold_label']][:5]

,sentence1,sentence2,gold_label
0,A person on a horse jumps over a broken down a...,A person is training his horse for a competition.,neutral
1,A person on a horse jumps over a broken down a...,"A person is at a diner, ordering an omelette.",contradiction
2,A person on a horse jumps over a broken down a...,"A person is outdoors, on a horse.",entailment
3,Children smiling and waving at camera,They are smiling at their parents,neutral
4,Children smiling and waving at camera,There are children present,entailment


## Model

BERT (Bidirectional Encoder Representations from Transformers) is a state-of-the-art approach to various NLP tasks. It uses a Transformer architecture and relies heavily on the concept of pre-training.

We'll use a pre-trained BERT-base model, provided in the huggingface [transformers](https://github.com/huggingface/transformers) repo.
It gives us a PyTorch implementation for the classic BERT architecture, as well as a tokenizer and weights, pre-trained on a public English corpus (Wikipedia).

Please follow these [installation instructions](https://github.com/huggingface/transformers#installation) before proceeding.

In [ ]:
from transformers import BertConfig, BertTokenizer, BertForSequenceClassification
def create_tokenizer_and_model():

    model_name = "bert-base-cased"
    config = BertConfig.from_pretrained(
        model_name,
        num_labels=3,
    )
    tokenizer = BertTokenizer.from_pretrained(
        "bert-base-cased",
        do_lower_case=False,
    )
    model = BertForSequenceClassification.from_pretrained(
        "bert-base-cased",
        config=config,
    )
    return tokenizer, model

tokenizer, model = create_tokenizer_and_model()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


The model has the following structure. It uses a combination of word, positional and token *embeddings* to create a sequence representation, then passes the data through 12 *transformer encoders* and finally uses a *linear classifier* to produce the final label.
As the model is already pre-trained and we only plan to fine-tune a few upper layers, we want to freeze all layers, except for the last encoder and above (`BertPooler` and `Classifier`).

In [ ]:
def freeze_non_trainable_parameters(model):
    trainable_layers = [model.bert.encoder.layer[-1], model.bert.pooler, model.classifier]
    total_params = 0
    trainable_params = 0

    for p in model.parameters():
            p.requires_grad = False
            total_params += p.numel()

    for layer in trainable_layers:
        for p in layer.parameters():
            p.requires_grad = True
            trainable_params += p.numel()

    print(f"Total parameters count: {total_params:,}") # ~108M
    print(f"Trainable parameters count: {trainable_params:,}") # ~7M

Thus, by using a pre-trained model we reduce the number of trainable params from over 100 million to just above 7.5 million. This will help both performance and convergence with added noise.

## Prepare the data

Before we begin training, we need to preprocess the data and convert it to the format our model expects.

(Note: it'll take 5-10 minutes to run on a laptop)

In [ ]:
import torch
import torch.nn as nn
import transformers
from torch.utils.data import TensorDataset
from transformers.data.processors.utils import InputExample
from transformers.data.processors.glue import glue_convert_examples_to_features

In [ ]:
LABEL_LIST = ['contradiction', 'entailment', 'neutral']
MAX_SEQ_LENGHT = 128




def _create_examples(df, set_type):
    """ Convert raw dataframe to a list of InputExample. Filter malformed examples
    """
    examples = []
    for index, row in df.iterrows():
        if row['gold_label'] not in LABEL_LIST:
            continue
        if not isinstance(row['sentence1'], str) or not isinstance(row['sentence2'], str):
            continue

        guid = f"{index}-{set_type}"
        examples.append(
            InputExample(guid=guid, text_a=row['sentence1'], text_b=row['sentence2'], label=row['gold_label']))
    return examples

def _df_to_features(df, set_type):
    """ Pre-process text. This method will:
    1) tokenize inputs
    2) cut or pad each sequence to MAX_SEQ_LENGHT
    3) convert tokens into ids

    The output will contain:
    `input_ids` - padded token ids sequence
    `attention mask` - mask indicating padded tokens
    `token_type_ids` - mask indicating the split between premise and hypothesis
    `label` - label
    """
    examples = _create_examples(df, set_type)

    #backward compatibility with older transformers versions
    legacy_kwards = {}
    from packaging import version
    if version.parse(transformers.__version__) < version.parse("2.9.0"):
        legacy_kwards = {
            "pad_on_left": False,
            "pad_token": tokenizer.convert_tokens_to_ids([tokenizer.pad_token])[0],
            "pad_token_segment_id": 0,
        }

    return glue_convert_examples_to_features(
        examples=examples,
        tokenizer=tokenizer,
        label_list=LABEL_LIST,
        max_length=MAX_SEQ_LENGHT,
        output_mode="classification",
        **legacy_kwards,
    )

def _features_to_dataset(features):
    """ Convert features from `_df_to_features` into a single dataset
    """
    all_input_ids = torch.tensor([f.input_ids for f in features], dtype=torch.long)
    all_attention_mask = torch.tensor(
        [f.attention_mask for f in features], dtype=torch.long
    )
    all_token_type_ids = torch.tensor(
        [f.token_type_ids for f in features], dtype=torch.long
    )
    all_labels = torch.tensor([f.label for f in features], dtype=torch.long)
    dataset = TensorDataset(
        all_input_ids, all_attention_mask, all_token_type_ids, all_labels
    )

    return dataset

train_features = _df_to_features(df_train, "train")
test_features = _df_to_features(df_test, "test")

train_dataset = _features_to_dataset(train_features)
test_dataset = _features_to_dataset(test_features)

## Choosing batch size

Let's talk about batch sizes for a bit.

In addition to all the considerations you normally take into account when choosing batch size, training models with DP adds another one - privacy cost.

Because of the threat model we assume and the way we add noise to the gradients, larger batch sizes (to a certain extent) generally help convergence. We add the same amount of noise to each gradient update (scaled to the norm of one sample in the batch) regardless of the batch size. What this means is that as the batch size increases, the relative amount of noise added decreases. while preserving the same epsilon guarantee.

You should, however, keep in mind that increasing batch size has its price in terms of epsilon, which grows at `O(sqrt(batch_size))` as we train (therefore larger batches make it grow faster). The good strategy here is to experiment with multiple combinations of `batch_size` and `noise_multiplier` to find the one that provides the best possible quality at acceptable privacy guarantee.

There's another side to this - memory. Opacus computes and stores *per sample* gradients, so for every normal gradient, Opacus will store `n=batch_size` per-sample gradients on each step, thus increasing the memory footprint by at least `O(batch_size)`. In reality, however, the peak memory requirement is `O(batch_size^2)` compared to a non-private model. This is because some intermediate steps in per sample gradient computation involve operations on two matrices, each with batch_size as one of the dimensions.

The good news is, we can pick the most appropriate batch size, regardless of memory constraints. Opacus has built-in support for *virtual* batches. Using it we can separate physical steps (gradient computation) and logical steps (noise addition and parameter updates): use larger batches for training, while keeping memory footprint low. Below we will specify two constants:

- `MAX_PHYSICAL_BATCH_SIZE` defines the maximum batch size we can afford from a memory standpoint, and only affects computation speed
- `BATCH_SIZE`, on the other hand, will affect only convergence and privacy guarantee.



In [ ]:
BATCH_SIZE = 16
MAX_PHYSICAL_BATCH_SIZE = 8

In [ ]:
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler
from opacus.utils.uniform_sampler import UniformWithReplacementSampler

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_dataset, sampler=SequentialSampler(test_dataset), batch_size=BATCH_SIZE)

## Training

In [ ]:
# Move the model to appropriate device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Set the model to train mode (HuggingFace models load in eval mode)
model = model.train()
# Define optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, eps=1e-8)

Let’s now define the evaluation cycle.

In [ ]:
import numpy as np
from tqdm.notebook import tqdm

def accuracy(preds, labels):
    return (preds == labels).mean()

# define evaluation cycle
def evaluate(model):
    model.eval()

    loss_arr = []
    accuracy_arr = []

    for batch in test_dataloader:
        batch = tuple(t.to(device) for t in batch)

        with torch.no_grad():
            inputs = {'input_ids':      batch[0],
                      'attention_mask': batch[1],
                      'token_type_ids': batch[2],
                      'labels':         batch[3]}

            outputs = model(**inputs)
            loss, logits = outputs[:2]

            preds = np.argmax(logits.detach().cpu().numpy(), axis=1)
            labels = inputs['labels'].detach().cpu().numpy()

            loss_arr.append(loss.item())
            accuracy_arr.append(accuracy(preds, labels))

    model.train()
    return np.mean(loss_arr), np.mean(accuracy_arr)

Next, we will define and attach PrivacyEngine. There are two parameters you need to consider here:

- `noise_multiplier`. It defines the trade-off between privacy and accuracy. Adding more noise will provide stronger privacy guarantees, but will also hurt model quality.  In this run, the PrivacyEngine will determine this value based on the target values of `EPSILON`, `DELTA`, and `EPOCHS`.  For the default settings, this will set `noise_multiplier` to about 0.4.
- `max_grad_norm`. Defines the maximum magnitude of L2 norms to which we clip per sample gradients. There is a bit of tug of war with this threshold: on the one hand, a low threshold means that we will clip many gradients, hurting convergence, so we might be tempted to raise it. However, recall that we add noise with `std=noise_multiplier * max_grad_norm` so we will pay for the increased threshold with more noise. In most cases you can rely on the model being quite resilient to clipping (after the first few iterations your model will tend to adjust so that its gradients stay below the clipping threshold), so you can often just keep the default value (`=1.0`) and focus on tuning `batch_size` and `noise_multiplier` instead. That being said, sometimes clipping hurts the model so it may be worth experimenting with different clipping thresholds, like we are doing in this tutorial.

These two parameters define the scale of the noise we add to gradients: the noise will be sampled from a Gaussian distribution with `std=noise_multiplier * max_grad_norm`.


In [ ]:
results: dict = {}
EPOCHS = 2
LOGGING_INTERVAL = 1000

In [ ]:
import time
import json

Now we can train the model.

In [ ]:
tokenizer, model = create_tokenizer_and_model()
freeze_non_trainable_parameters(model)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

model = model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, eps=1e-8)
training_start_time = time.time()
epoch_arr = []
step_arr = []
train_loss_arr = []
eval_loss_arr = []
eval_accuracy_arr = []

for epoch in range(1, EPOCHS+1):

    losses = []

    for step, batch in enumerate(tqdm(memory_safe_data_loader)):
        optimizer.zero_grad()

        batch = tuple(t.to(device) for t in batch)
        inputs = {'input_ids':      batch[0],
                'attention_mask': batch[1],
                'token_type_ids': batch[2],
                'labels':         batch[3]}

        outputs = model(**inputs)

        loss = outputs[0]
        loss.backward()
        losses.append(loss.item())

        optimizer.step()

        if step > 0 and step % LOGGING_INTERVAL == 0:
            train_loss = np.mean(losses)
            eps = privacy_engine.get_epsilon(DELTA)

            eval_loss, eval_accuracy = evaluate(model)

            epoch_arr.append(epoch)
            step_arr.append(step)
            train_loss_arr.append(step)
            eval_loss_arr.append(eval_loss)
            eval_accuracy_arr.append(eval_accuracy)

            print(
              f"Epoch: {epoch} | "
              f"Step: {step} | "
              f"Train loss: {train_loss:.3f} | "
              f"Eval loss: {eval_loss:.3f} | "
              f"Eval accuracy: {eval_accuracy:.3f} | "
            )


training_end_time = time.time()
total_training_time = training_end_time - training_start_time
print(f"Total training time for EPSILON={EPSILON}: {total_training_time:.2f} seconds")

results = dict(epoch_arr=epoch_arr,step_arr=step_arr,train_loss_arr=train_loss_arr,eval_loss_arr=eval_loss_arr, eval_accuracy_arr=eval_accuracy_arr, training_time=total_training_time)

with open('results.json', 'w') as f:
    f.write(json.dumps(results, indent=4))


Training EPSILON: 10


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters count: 108,312,579
Trainable parameters count: 7,680,771


  0%|          | 0/68671 [00:00<?, ?it/s]

Epoch: 1 | Step: 1000 | Train loss: 1.182 | Eval loss: 1.311 | Eval accuracy: 0.434 | ɛ: 3.88
Epoch: 1 | Step: 2000 | Train loss: 1.228 | Eval loss: 1.259 | Eval accuracy: 0.449 | ɛ: 4.60
Epoch: 1 | Step: 3000 | Train loss: 1.263 | Eval loss: 1.308 | Eval accuracy: 0.460 | ɛ: 5.01
Epoch: 1 | Step: 4000 | Train loss: 1.260 | Eval loss: 1.239 | Eval accuracy: 0.464 | ɛ: 5.29
Epoch: 1 | Step: 5000 | Train loss: 1.276 | Eval loss: 1.215 | Eval accuracy: 0.513 | ɛ: 5.51
Epoch: 1 | Step: 6000 | Train loss: 1.292 | Eval loss: 1.486 | Eval accuracy: 0.520 | ɛ: 5.69
Epoch: 1 | Step: 7000 | Train loss: 1.313 | Eval loss: 1.571 | Eval accuracy: 0.543 | ɛ: 5.84
Epoch: 1 | Step: 8000 | Train loss: 1.361 | Eval loss: 1.639 | Eval accuracy: 0.574 | ɛ: 5.97
Epoch: 1 | Step: 9000 | Train loss: 1.409 | Eval loss: 1.627 | Eval accuracy: 0.587 | ɛ: 6.08
Epoch: 1 | Step: 10000 | Train loss: 1.447 | Eval loss: 1.666 | Eval accuracy: 0.594 | ɛ: 6.19
Epoch: 1 | Step: 11000 | Train loss: 1.477 | Eval loss: 1.6

...

Epoch: 2 | Step: 6000 | Train loss: 1.772 | Eval loss: 1.710 | Eval accuracy: 0.747 | ɛ: 77.28
Epoch: 2 | Step: 7000 | Train loss: 1.777 | Eval loss: 1.668 | Eval accuracy: 0.748 | ɛ: 77.61
Epoch: 2 | Step: 8000 | Train loss: 1.774 | Eval loss: 1.714 | Eval accuracy: 0.750 | ɛ: 77.93
Epoch: 2 | Step: 9000 | Train loss: 1.769 | Eval loss: 1.688 | Eval accuracy: 0.748 | ɛ: 78.26
Epoch: 2 | Step: 10000 | Train loss: 1.768 | Eval loss: 1.716 | Eval accuracy: 0.749 | ɛ: 78.58
Epoch: 2 | Step: 11000 | Train loss: 1.772 | Eval loss: 1.758 | Eval accuracy: 0.749 | ɛ: 78.90
Epoch: 2 | Step: 12000 | Train loss: 1.775 | Eval loss: 1.768 | Eval accuracy: 0.750 | ɛ: 79.23
Epoch: 2 | Step: 13000 | Train loss: 1.778 | Eval loss: 1.701 | Eval accuracy: 0.751 | ɛ: 79.55
Epoch: 2 | Step: 14000 | Train loss: 1.774 | Eval loss: 1.707 | Eval accuracy: 0.751 | ɛ: 79.87
Epoch: 2 | Step: 15000 | Train loss: 1.771 | Eval loss: 1.669 | Eval accuracy: 0.754 | ɛ: 80.19
Epoch: 2 | Step: 16000 | Train loss: 1.771 |

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters count: 108,312,579
Trainable parameters count: 7,680,771


  0%|          | 0/68671 [00:00<?, ?it/s]

Epoch: 1 | Step: 1000 | Train loss: 1.116 | Eval loss: 1.130 | Eval accuracy: 0.422 | ɛ: 0.02
Epoch: 1 | Step: 2000 | Train loss: 1.140 | Eval loss: 1.137 | Eval accuracy: 0.432 | ɛ: 0.02
Epoch: 1 | Step: 3000 | Train loss: 1.161 | Eval loss: 1.248 | Eval accuracy: 0.439 | ɛ: 0.02
Epoch: 1 | Step: 4000 | Train loss: 1.186 | Eval loss: 1.286 | Eval accuracy: 0.452 | ɛ: 0.02
Epoch: 1 | Step: 5000 | Train loss: 1.219 | Eval loss: 1.255 | Eval accuracy: 0.449 | ɛ: 0.02
Epoch: 1 | Step: 6000 | Train loss: 1.232 | Eval loss: 1.310 | Eval accuracy: 0.467 | ɛ: 0.03
Epoch: 1 | Step: 7000 | Train loss: 1.243 | Eval loss: 1.168 | Eval accuracy: 0.487 | ɛ: 0.03
Epoch: 1 | Step: 8000 | Train loss: 1.246 | Eval loss: 1.202 | Eval accuracy: 0.488 | ɛ: 0.03
Epoch: 1 | Step: 9000 | Train loss: 1.246 | Eval loss: 1.200 | Eval accuracy: 0.506 | ɛ: 0.03
Epoch: 1 | Step: 10000 | Train loss: 1.248 | Eval loss: 1.209 | Eval accuracy: 0.507 | ɛ: 0.03
Epoch: 1 | Step: 11000 | Train loss: 1.249 | Eval loss: 1.2

In [ ]:

save_dir = f"saved_models/bert-base-uncased-snli"
os.makedirs(save_dir, exist_ok=True)

model_save_path = os.path.join(save_dir, "model.pt")
tokenizer_save_path = os.path.join(save_dir, "tokenizer")

torch.save(model.state_dict(), model_save_path)
tokenizer.save_pretrained(tokenizer_save_path)

print(f"Model and tokenizer saved to {save_dir}")


In [ ]:
import json

with open('results.json', 'w') as f:
    f.write(json.dumps(results, indent=4))


For the test accuracy, after training for three epochs you should expect something close to the results below.

You can see that we can achieve quite strong privacy guarantee at epsilon=7.5 with a moderate accuracy cost of 11 percentage points compared to non-private model trained in a similar setting (upper layers only) and 16 points compared to best results we were able to achieve using the same architecture.

*NB: When not specified, DP-SGD is trained with upper layers only*

| Model | Noise multiplier | Batch size | Accuracy | Epsilon |
| --- | --- | --- | --- | --- |
| no DP, train full model | N/A | 32 | 90.1% | N/A |
| no DP, train upper layers only | N/A | 32 | 85.4% | N/A |
| DP-SGD | 1.0 | 32 | 70.5% | 0.7 |
| **DP-SGD (this tutorial)** | **0.4** | **32** | **74.3%** | **7.5** |
| DP-SGD | 0.3 | 32 | 75.8% | 20.7 |
| DP-SGD | 0.1 | 32 | 78.3% | 2865 |
| DP-SGD | 0.4 | 8 | 67.3% | 5.9 |